In [1]:
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pickle

from pathlib import Path
#from xps_datagen.NNDataBuilderPeter import gen_cs
from networks.NNModels import build_2D_cnn, build_2D_cnn_functional  

import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

from datetime import datetime

In [2]:
NUM_PEAKS = 1
train_type = 'Intensity'
batch_size = 250
print(tf.__version__)
#print(tf.executing_eagerly())

2.19.0


In [3]:
filename_format = '{}records{}.tfrecords'
records_per_file = 10000
num_files = 10
validation_percentage = 15/100
filenames = [filename_format.format(records_per_file, i+1) for i in range(num_files)]

file_dir = "Y:/{}PeakC1sData".format(NUM_PEAKS)
save_path = "C:/Users/kxl61517/Documents/xsw_models"

file_paths = ['{}/{}'.format(file_dir, filename) for filename in filenames]
train_file_paths, val_file_paths = np.split(file_paths, [int(num_files*(1-validation_percentage))])

train_size = len(train_file_paths) * records_per_file
val_size = len(val_file_paths) * records_per_file


In [24]:
features_dict = {
    'xsw_curve': tf.io.FixedLenFeature(shape = (504, 200, 1), dtype = tf.float32),
    'cs_norm': tf.io.FixedLenFeature(shape = (46, 1), dtype = tf.float32),
}

train_files = tf.data.Dataset.from_tensor_slices(train_file_paths)
val_files = tf.data.Dataset.from_tensor_slices(val_file_paths)
block_length = 10
train_dataset = train_files.interleave(lambda filename: tf.data.TFRecordDataset(filename).prefetch(num_files * block_length))
val_dataset = val_files.interleave(lambda filename: tf.data.TFRecordDataset(filename))
train_dataset = train_dataset.map(map_func = lambda serialized: tf.io.parse_single_example(serialized=serialized, features = features_dict))
val_dataset = val_dataset.map(map_func = lambda serialized: tf.io.parse_single_example(serialized=serialized, features = features_dict))


train_dataset = train_dataset.map(map_func = lambda features:
                        (features['xsw_curve'],
                        #(tf.gather(features['cs_norm'], 14) * (tf.reduce_max(tf.reduce_max(features['xsw_curve'], axis = 1))))
                        ), num_parallel_calls=tf.data.AUTOTUNE
                      )

#val_dataset = val_dataset.map(map_func = lambda features:
#                        (features['xsw_curve'], tf.gather(features['cs_norm'], 15) * tf.reduce_max(features['xsw_curve'])
#                        )
#                        , num_parallel_calls=tf.data.AUTOTUNE)
#train_dataset = train_dataset.batch(batch_size)
#val_dataset = val_dataset.batch(batch_size)


In [25]:
print(np.array(
    next(
    train_dataset.as_numpy_iterator()
    )
    ).shape
)


(1, 504, 200, 1)


xsw_curve shape: (504, 200)
504 comes from modulation, 200 is the number of data points in the spectra
need to find the location of the highest element in modulation, i, and the location of each peak in the spectra, j, and then xsw_curve[i, j] is the xsw peak intensity?
j - 200 * peak pos param from cs_norm
i - modulation not saved in multi peak case -  

In [8]:
model, hyper_params = build_2D_cnn_functional(dense_neurons = [512, 128, NUM_PEAKS])
model.summary()

{'output_0': 'mae', 'output_1': 'mae'}
{'output_0': ['mse'], 'output_1': ['mse']}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 504, 200,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 504, 200,  │        416 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 504, 200,  │      6,416 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 252, 100,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 252, 100,  │     12,832 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 126, 50,   │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 126, 50,   │     38,448 │ max_pooling2d_1[… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 63, 25,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 63, 25,    │     76,864 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 32, 13,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 13,    │    128,080 │ max_pooling2d_3[… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 16, 7, 80) │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 7, 96) │    192,096 │ max_pooling2d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 8, 4, 96)  │          0 │ conv2d_6[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3072)      │          0 │ max_pooling2d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,573,376 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     65,664 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 2,094,450 (7.99 MB)

 Trainable params: 2,094,450 (7.99 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
currenttime = datetime.now()
model_suffix = '-' + str(currenttime.strftime("%d%m%H"))
model_name = '{}/{}PeakC1s{}Model{}.keras'.format(save_path, NUM_PEAKS, train_type, model_suffix)
history_filename = '{}/{}TrainHistory{}.pkl'.format(save_path, train_type, model_suffix)

#tf.profiler.experimental.start('logdir')
history = model.fit(train_dataset, epochs = 1, steps_per_epoch=10, validation_data = val_dataset, validation_steps = 10)
#tf.profiler.experimental.stop()
#model.save(model_name)

#with open(history_filename, 'wb') as file_pi:
#  pickle.dump(history.history, file_pi)

c:\Users\kxl61517\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\optimizers\base_optimizer.py:678: UserWarning: Gradients do not exist for variables ['kernel', 'bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 443ms/step - loss: 0.2471 - output_0_mse: 0.0892 - val_loss: 0.2081 - val_output_0_mse: 0.0567


In [10]:
#model.predict(val_dataset)
history.history

{'loss': [0.236233651638031],
 'output_0_mse': [0.07918120175600052],
 'val_loss': [0.20806476473808289],
 'val_output_0_mse': [0.05672285705804825]}